# Retail Detector — RF-DETR Fine-Tuning on Google Colab T4

**Hardware:** Google Colab T4 GPU (16 GB VRAM)  
**Model:** RF-DETR Base (DINOv2-B backbone)  
**Classes:** bottle · box · can · bag

Run cells top-to-bottom. Cell 1 will automatically install dependencies and restart the session once (to apply the NumPy version downgrade). After the restart, run Cell 1 again, then proceed with the rest of the cells.

In [ ]:
# ── Cell 1: Install dependencies and setup environment ────────────────────────
import os, sys, subprocess

setup_flag = "/content/.setup_done"

if os.path.exists(setup_flag):
    print("✅ Setup flag found. Environment has already been prepared.")
    try:
        import numpy as np
        import torch
        import supervision as sv
        import rfdetr
        import faster_coco_eval
        
        print("=" * 50)
        print("PyTorch          :", torch.__version__)
        print("CUDA             :", torch.cuda.is_available())
        for i in range(torch.cuda.device_count()):
            print(f"  GPU {i}         :", torch.cuda.get_device_name(i))
        print("NumPy            :", np.__version__)
        print("Supervision      :", sv.__version__)
        print("rfdetr           :", getattr(rfdetr, "__version__", "OK"))
        print("faster-coco-eval :", faster_coco_eval.__version__, "✅")
        print("=" * 50)
        print("All packages are ready!")
    except ImportError as e:
        print(f"❌ Error importing packages: {e}")
        print("If imports are failing, please manually click 'Runtime' -> 'Restart session' from the top menu and run this cell again.")
else:
    print("Initializing environment: Installing dependencies with NumPy 1.x...")
    pkgs = ["numpy>=1.24.0,<2.0.0", "rfdetr", "supervision", "faster-coco-eval"]
    try:
        # Use --force-reinstall to clean up any corrupted package states
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall"] + pkgs, check=True)
        
        # Write setup flag to prevent infinite crash/restart loops
        with open(setup_flag, "w") as f:
            f.write("done")
            
        print("\n🔄 Packages installed. Restarting the session to apply changes...")
        print("👉 AFTER THE SESSION RESTARTS, PLEASE RUN THIS CELL AGAIN.")
        os.kill(os.getpid(), 9)
    except Exception as e:
        print(f"❌ Installation failed: {e}")

In [ ]:
# ── Cell 1.5: Mount Google Drive ──────────────────────────────────────────────
# This ensures we can read your uploaded dataset and save models persistently.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: VRAM check + safe batch-size selection ────────────────────────────
#
# FIX #1 — The original plan used batch_size=8 which OOMs on a single T4.
# RF-DETR Base with DINOv2-B at imgsz=640 uses ~1.8 GB per sample.
# We probe available VRAM and pick the largest safe batch size automatically.
# Effective batch (batch_size × grad_accum_steps) is kept at 32 throughout.

if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Enable GPU: Settings → Accelerator → GPU T4 x2')

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM on device 0: {vram_gb:.1f} GB')

# Conservative mapping — leaves ~20% headroom for gradients + optimizer state
if vram_gb >= 24:          # A100 / V100 32 GB
    BATCH_SIZE      = 8
    GRAD_ACCUM      = 4
elif vram_gb >= 15:        # T4 16 GB — the Kaggle default
    BATCH_SIZE      = 4
    GRAD_ACCUM      = 8
elif vram_gb >= 10:        # P100 12 GB
    BATCH_SIZE      = 2
    GRAD_ACCUM      = 16
else:                      # anything smaller — CPU fallback
    BATCH_SIZE      = 1
    GRAD_ACCUM      = 32

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
print(f'Selected  batch_size={BATCH_SIZE}, grad_accum={GRAD_ACCUM}  '
      f'→ effective batch={EFFECTIVE_BATCH}')

In [ ]:
# ── Cell 3: Dataset path configuration ───────────────────────────────────────────
# Update DATASET_ROOT to point to the folder where you uploaded the dataset in Google Drive.
# e.g., if you uploaded the dataset folder to the root of your Drive, it might be:
# DATASET_ROOT = '/content/drive/MyDrive/retail-shelf-coco'

DATASET_ROOT = '/content/drive/MyDrive/Colab _Notebooks/Retailshelfdetector.v6i.coco(1)' # <--- CHANGE THIS IF NEEDED

if not os.path.exists(DATASET_ROOT):
    print(f'\n⚠️ ERROR: Directory not found -> {DATASET_ROOT}')
    print('Please check your Google Drive and update the DATASET_ROOT path.')
    raise RuntimeError('Dataset path not found.')

print(f'Dataset root: {DATASET_ROOT}')
for split in ['train', 'valid', 'test']:
    ann = os.path.join(DATASET_ROOT, split, '_annotations.coco.json')
    imgs = os.path.join(DATASET_ROOT, split, 'images')
    img_count = len(os.listdir(imgs)) if os.path.isdir(imgs) else 0
    print(f'  {split}: {img_count} images  |  annotation file: {os.path.exists(ann)}')


In [ ]:
# ── Cell 4: Class balance check ───────────────────────────────────────────────
# FIX #4 — Threshold changed from 0.5× to 60-instance minimum.
# A 2:1 ratio is fine; fewer than 60 instances means the class will underfit.

from collections import Counter

def check_balance(split='train', min_instances=60):
    ann_path = os.path.join(DATASET_ROOT, split, '_annotations.coco.json')
    with open(ann_path) as f:
        coco = json.load(f)
    cats   = {c['id']: c['name'] for c in coco['categories']}
    counts = Counter(cats[a['category_id']] for a in coco['annotations'])
    print(f'\n{split} split class distribution:')
    for cls, n in sorted(counts.items(), key=lambda x: -x[1]):
        bar    = '█' * (n // max(1, max(counts.values()) // 30))
        warn   = '  ⚠️  BELOW MINIMUM' if n < min_instances else ''
        print(f'  {cls:<10s}: {n:4d}  {bar}{warn}')
    under = [cls for cls, n in counts.items() if n < min_instances]
    if under:
        print(f'\n⚠️  Classes with < {min_instances} instances: {under}')
        print('   Collect more images for these classes before training.')
    else:
        print(f'\n✅ All classes have ≥ {min_instances} instances.')
    return counts

counts = check_balance('train', min_instances=60)

In [ ]:
# ── Cell 5: 1-epoch smoke test ─────────────────────────────────────────────────
# FIX #1 (continued) — Always smoke-test before the full run.
# This validates: dataset format, batch size fits in VRAM, model loads correctly.
# If this cell completes without OOM, proceed to Cell 6 (full training).

from rfdetr import RFDETRBase

print('Running 1-epoch smoke test …')
smoke_model = RFDETRBase()
try:
    smoke_model.train(
        dataset_dir=DATASET_ROOT,
        epochs=1,
        batch_size=BATCH_SIZE,
        grad_accum_steps=GRAD_ACCUM,
        lr=1e-4,
        lr_encoder=1e-5,
        output_dir='/content/smoke_test',
    )
    print('\n✅ Smoke test passed — proceeding to full training.')
except torch.cuda.OutOfMemoryError:
    # Auto-recover: halve batch size and double grad accum
    BATCH_SIZE  = max(1, BATCH_SIZE // 2)
    GRAD_ACCUM  = GRAD_ACCUM * 2
    EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
    print(f'\n⚠️  OOM! Auto-reduced to batch_size={BATCH_SIZE}, '
          f'grad_accum={GRAD_ACCUM}  (effective={EFFECTIVE_BATCH})')
    print('Re-run this cell to re-try the smoke test.')
    raise
del smoke_model
torch.cuda.empty_cache()

In [ ]:
# ── Cell 6: Full training ─────────────────────────────────────────────────────
#
# Two-LR rationale (document this in your write-up):
#   lr=1e-4       → detection head (randomly initialised, needs to learn fast)
#   lr_encoder=1e-5 → DINOv2 backbone (pretrained on 142M images, nudge gently)
# Using the same LR for both either overwrites backbone features (too high)
# or prevents the head from learning (too low).

OUT_DIR = '/content/drive/MyDrive/rfdetr_runs'

model = RFDETRBase()
t0 = time.time()

model.train(
    dataset_dir      = DATASET_ROOT,
    epochs           = 60,
    batch_size       = BATCH_SIZE,       # auto-selected in Cell 2
    grad_accum_steps = GRAD_ACCUM,
    lr               = 1e-4,            # detection head
    lr_encoder       = 1e-5,            # DINOv2 backbone
    checkpoint_interval = 5,
    output_dir       = OUT_DIR,
)

elapsed = (time.time() - t0) / 60
print(f'\nTraining finished in {elapsed:.1f} min')
print('Checkpoints:')
for f in sorted(os.listdir(OUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6
    print(f'  {f}  ({size_mb:.0f} MB)')

In [ ]:
# ── Cell 7: Loss curve check ──────────────────────────────────────────────────
# Healthy: train loss ↓ steadily, val loss tracks it.
# Overfitting signal: val loss rises while train loss keeps falling after ~epoch 20.
# If you see overfitting: add dropout=0.1 and resume from checkpoint at epoch 20.

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

log_path = os.path.join(OUT_DIR, 'log.txt')   # rfdetr writes a log.txt
if os.path.exists(log_path):
    train_losses, val_losses, epochs_axis = [], [], []
    with open(log_path) as f:
        for line in f:
            try:
                entry = json.loads(line)
                if 'train_loss' in entry:
                    epochs_axis.append(entry.get('epoch', len(epochs_axis)))
                    train_losses.append(entry['train_loss'])
                    val_losses.append(entry.get('val_loss', None))
            except Exception:
                pass

    if train_losses:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(epochs_axis, train_losses, label='Train loss', color='#3b82f6')
        valid = [(e, v) for e, v in zip(epochs_axis, val_losses) if v is not None]
        if valid:
            ax.plot(*zip(*valid), label='Val loss', color='#f59e0b', linestyle='--')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.set_title('Training Loss Curve')
        ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig('/content/drive/MyDrive/rfdetr_runs/loss_curve.png', dpi=150)
        plt.show()
        print('Loss curve saved to /content/drive/MyDrive/rfdetr_runs/loss_curve.png')
    else:
        print('Could not parse loss values from log.txt')
else:
    print(f'log.txt not found at {log_path}')
    print('Available files:', os.listdir(OUT_DIR))

In [ ]:
# ── Cell 8: Export best checkpoint + ONNX ──────────────────────────────────────
# FIX #7 — Export to ONNX for Intel Arc local inference via OpenVINO.
# The .pth file is for reference; the .onnx is what runs fast locally.

import shutil

# Find best checkpoint
ckpts = sorted(
    [f for f in os.listdir(OUT_DIR) if f.endswith('.pth')],
    key=lambda f: os.path.getmtime(os.path.join(OUT_DIR, f))
)
best_ckpt = os.path.join(OUT_DIR, ckpts[-1]) if ckpts else None
print(f'Best checkpoint: {best_ckpt}')

# Copy to a clean output path for easy download
os.makedirs('/content/drive/MyDrive/rfdetr_export', exist_ok=True)
if best_ckpt:
    shutil.copy(best_ckpt, '/content/drive/MyDrive/rfdetr_export/best_checkpoint.pth')

# ── ONNX export ────────────────────────────────────────────────────────────
# Check if rfdetr supports .export() — API varies by version
model_for_export = RFDETRBase(pretrain_weights=best_ckpt)
onnx_path = '/content/drive/MyDrive/rfdetr_export/retail_detector.onnx'

try:
    # rfdetr >= 0.2.0 exposes model.export()
    model_for_export.export(onnx_path)
    print(f'✅ ONNX exported: {onnx_path}')
except AttributeError:
    # Fallback: manual torch.onnx.export
    print('rfdetr.export() not available — using torch.onnx.export fallback')
    import torch
    dummy_input = torch.zeros(1, 3, 640, 640).cuda()
    # RF-DETR wraps a standard nn.Module; access via model.model
    net = model_for_export.model if hasattr(model_for_export, 'model') else model_for_export
    net.eval()
    try:
        torch.onnx.export(
            net.cpu(),
            dummy_input.cpu(),
            onnx_path,
            opset_version=17,
            input_names=['image'],
            output_names=['boxes', 'scores', 'labels'],
            dynamic_axes={'image': {0: 'batch'}},
        )
        print(f'✅ ONNX exported (fallback): {onnx_path}')
    except Exception as e:
        print(f'⚠️  ONNX export failed: {e}')
        print('   Use best_checkpoint.pth for local inference instead.')

print('\nFiles ready to download from /content/drive/MyDrive/rfdetr_export/:')
for f in os.listdir('/content/drive/MyDrive/rfdetr_export'):
    mb = os.path.getsize(f'/content/drive/MyDrive/rfdetr_export/{f}') / 1e6
    print(f'  {f}  ({mb:.0f} MB)')